# TMDB 1.4M Movie Embeddings — Qwen3-0.6B on T4 (v2)

**Runtime → Change runtime type → T4 GPU → Save**

### What changed from v1

| Change | Why |
|--------|-----|
| **HDF5 output** | One .h5 file per shard (embeddings + ids + rows). No .csv sidecar. |
| **Asymmetric prompts** | Movies get raw text. Only search queries get instructions. Matches Qwen3 official docs (+1-5% retrieval). |
| **Richer movie text** | Year, original_title, runtime, votes, country added. No more duplicate vectors. |
| **tmdb_id in text** | Prevents identical embeddings from rows with identical empty fields. |

### Colab free tier constraints
- **~5 hours/day** GPU usage limit — 3-4 sessions to finish all 1.4M rows
- **~12.7 GB RAM** — notebook stays under ~2 GB peak
- **~100 GB scratch disk** (`/content/`) — outputs saved every 20K rows
- **Resume via manifest.json** — upload zip from last session, run Cell 2, continue

### Per-session instructions
1. Cells 1→2→3→4→5 in order
2. Cell 6 processes until session ends (~5h)
3. **Cell 7** — zip & download before disconnect
4. Next session: upload zip → Cell 2 restores → run again

### Speed estimate
- **30-50 rows/sec** with BATCH=64, MAX_LEN=256
- **180K-300K rows per 5h session**
- **5-7 sessions** for all 1.4M rows
- **Output size**: ~4.3 GB total (768-dim float32 × 1.4M)


## Cell 1 — Install Dependencies


In [ ]:
!pip install -q -U transformers accelerate sentencepiece safetensors tokenizers h5py


## Cell 2 — Restore Previous Progress (skip on first run)

Upload `embeddings_progress.zip` from Cell 7 of your last session, then run this cell.


In [ ]:
import os, zipfile
from pathlib import Path

OUTPUT_DIR = "/content/embeddings"
ZIP_PATH   = "/content/embeddings_progress.zip"

if os.path.exists(ZIP_PATH):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall("/content/")
    print(f"Restored to {OUTPUT_DIR}/")
    h5_count = len(list(Path(OUTPUT_DIR).glob("*.h5")))
    print(f"Found {h5_count} shard files (.h5)")
else:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print("No previous progress zip found — starting fresh")
    print(f"Created: {OUTPUT_DIR}/")


## Cell 3 — Download TMDB Dataset (~632 MB CSV, 1.4M rows)


In [ ]:
CSV_PATH = "/content/TMDB_movie_dataset_v11.csv"
CSV_URL  = "https://huggingface.co/datasets/fukitweball/TMDB/resolve/main/TMDB_movie_dataset_v11.csv"

if os.path.exists(CSV_PATH):
    print(f"Already downloaded: {os.path.getsize(CSV_PATH)/1e6:.0f} MB")
else:
    !wget -q --show-progress -O "{CSV_PATH}" "{CSV_URL}"
    print(f"Done: {os.path.getsize(CSV_PATH)/1e6:.0f} MB")

# Quick row count for ETA
print("Counting rows...")
TOTAL_ROWS = sum(1 for _ in open(CSV_PATH, encoding="utf-8")) - 1
print(f"Total rows: {TOTAL_ROWS:,}")


## Cell 4 — KeepAlive (prevents idle disconnect)

Run once. Auto-clicks the Colab toolbar every 60s.


In [ ]:
from IPython.display import display, Javascript
display(Javascript("""
function ClickConnect() {
  const el = document.querySelector("colab-connect-button");
  if (el?.shadowRoot) {
    const btn = el.shadowRoot.querySelector("#connect");
    if (btn) btn.click();
  }
}
setInterval(ClickConnect, 60000);
console.log("[KeepAlive] Anti-disconnect active — runs every 60s");
"""))


## Cell 5 — GPU Check, Config & Load Model


In [ ]:
import torch, gc, json
assert torch.cuda.is_available(), "No GPU! Runtime → Change runtime type → T4 GPU"
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
!free -h

MODEL_ID   = "Qwen/Qwen3-Embedding-0.6B"

# Memory-safe settings for Colab free (12.7 GB RAM + 15 GB T4 VRAM)
BATCH_SIZE = 64       # Fits ~3 GB VRAM total (model + activations)
MAX_LEN    = 256      # Avg movie text is ~120-150 tokens after tokenization
OUT_DIM    = 768      # Matryoshka sweet spot (32-1024 supported)
SHARD_SIZE = 20_000   # Lower => less peak RAM, more frequent saves

# CSV dtype hints — cuts pandas RAM ~40%
CSV_DTYPES = {
    "id": "int32", "vote_average": "float32", "vote_count": "int32",
    "revenue": "float32", "runtime": "float32", "budget": "float32",
    "popularity": "float32", "adult": "bool", "release_date": "str",
}

print(f"BATCH={BATCH_SIZE}  MAX_LEN={MAX_LEN}  DIM={OUT_DIM}  SHARD={SHARD_SIZE:,}")
print(f"Total rows: {TOTAL_ROWS:,}  |  Expected shards: {(TOTAL_ROWS + SHARD_SIZE - 1) // SHARD_SIZE}")
print(f"Est. rows per 5h session: ~{(5*3600*35):,} (at 35 rows/s)")


In [ ]:
from transformers import AutoTokenizer, AutoModel

DEVICE = torch.device("cuda")
gc.collect()
torch.cuda.empty_cache()

print(f"Loading {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left", use_fast=True)
# bfloat16 — same VRAM as float16, better numeric stability on T4
model = AutoModel.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16)
model.eval().to(DEVICE)

params = sum(p.numel() for p in model.parameters()) / 1e6
vram   = torch.cuda.memory_allocated() / 1e9
print(f"Loaded: {params:.0f}M params | {vram:.2f} GB VRAM used")
print(f"Free VRAM: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.1f} GB")


## Cell 6 — Embed Movies (Resumable)

Streams CSV in 20K-row chunks. Each chunk becomes one `.h5` shard with:
- `/embeddings` — (N, 768) float32 unit-normalized vectors
- `/ids` — int32 TMDB movie IDs
- `/rows` — int32 CSV row positions

Skips shards already saved (detected via manifest.json). Saves manifest after every shard — survives crashes.


In [ ]:
import json, os, time, gc, h5py
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

# ── Safe value helper ─────────────────────────────────────────────────────
def safe(v) -> str:
    if v is None: return ""
    try:
        if pd.isna(v): return ""
    except (TypeError, ValueError): pass
    return str(v).strip()

# ── Text builder ──────────────────────────────────────────────────────────
# CAUTION: Do NOT add an "Instruct:" prefix here. Qwen3 expects asymmetric
# prompts — raw text for documents, instructions only on the query side.
# Adding instructions on both sides degrades retrieval by 1-5%.
# Also includes tmdb_id to prevent duplicate vectors when fields are empty.
def movie_text(row) -> str:
    movie_id  = safe(row.id)
    title     = safe(row.title)     or "Untitled"
    orig      = safe(row.original_title) or ""
    overview  = safe(row.overview)  or "none"
    genres    = safe(row.genres)    or "unknown"
    keywords  = safe(row.keywords)  or "none"
    tagline   = safe(row.tagline)   or "none"
    lang      = safe(row.original_language) or "unknown"
    country   = safe(row.production_countries) or "unknown"

    # Release year
    year = safe(row.release_date)[:4] if safe(row.release_date) else "unknown"

    # Runtime (with unit)
    rt = safe(row.runtime)
    runtime_str = f"{rt} min" if rt and rt != "unknown" else "unknown"

    # Rating with vote count for distinguishing popular films from obscure ones
    vote_str = ""
    vote_avg = safe(row.vote_average)
    vote_cnt = safe(row.vote_count)
    if vote_avg and vote_cnt:
        vote_str = f"{vote_avg}/10 ({vote_cnt} votes)"
    elif vote_avg:
        vote_str = f"{vote_avg}/10"
    else:
        vote_str = "unknown"

    parts = [f"Title: {title}", f"ID: {movie_id}"]
    if year and year != "unknown": parts.append(f"Year: {year}")
    if orig and orig.lower() != title.lower(): parts.append(f"Original title: {orig}")
    parts.append(f"Tagline: {tagline}")
    parts.append(f"Overview: {overview}")
    parts.append(f"Genres: {genres}")
    parts.append(f"Keywords: {keywords}")
    parts.append(f"Language: {lang}")
    parts.append(f"Country: {country}")
    parts.append(f"Runtime: {runtime_str}")
    parts.append(f"Rating: {vote_str}")

    return "\n".join(parts)

# ── Pooling (Qwen3 official last-token pool) ──────────────────────────────
def last_token_pool(hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    left_pad = (mask[:, -1].sum() == mask.shape[0])
    if left_pad:
        return hidden[:, -1]
    lengths = mask.sum(dim=1) - 1
    return hidden[torch.arange(hidden.shape[0], device=hidden.device), lengths]

# ── Embed one batch ───────────────────────────────────────────────────────
def embed_batch(texts: list[str]) -> np.ndarray:
    enc = tokenizer(
        texts, padding=True, truncation=True, max_length=MAX_LEN,
        return_tensors="pt",
    )
    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)
    del enc

    with torch.inference_mode():
        out  = model(input_ids=input_ids, attention_mask=attention_mask)
        vecs = last_token_pool(out.last_hidden_state, attention_mask)
        vecs = vecs[:, :OUT_DIM]
        vecs = F.normalize(vecs.float(), p=2, dim=1)

    result = vecs.cpu().numpy().astype(np.float32)
    del input_ids, attention_mask, out, vecs
    return result

# ── Embed entire shard with length-sorted batching ────────────────────────
# Sort by string length → similar-length texts batch together → less padding
def embed_shard(texts: list[str]) -> np.ndarray:
    n = len(texts)
    order = sorted(range(n), key=lambda i: len(texts[i]))
    sorted_texts = [texts[i] for i in order]

    all_vecs = [None] * n
    n_batches = (n + BATCH_SIZE - 1) // BATCH_SIZE

    for b in tqdm(range(n_batches), desc="  batches", leave=False):
        start = b * BATCH_SIZE
        end   = min(start + BATCH_SIZE, n)
        batch_texts   = sorted_texts[start:end]
        batch_indices = order[start:end]

        vecs = embed_batch(batch_texts)
        for local_i, orig_i in enumerate(batch_indices):
            all_vecs[orig_i] = vecs[local_i]
        del vecs

        if (b + 1) % 15 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    gc.collect()
    torch.cuda.empty_cache()

    arr = np.stack(all_vecs, axis=0)
    assert arr.shape == (n, OUT_DIM), f"Shape error: {arr.shape}"
    assert np.isfinite(arr).all(), "NaN/Inf in embeddings!"
    return arr

# ── Resume helper ─────────────────────────────────────────────────────────
def shard_exists(h5_path: Path, expected_rows: int) -> bool:
    if not h5_path.exists(): return False
    try:
        with h5py.File(h5_path, "r") as f:
            return (f["embeddings"].shape == (expected_rows, OUT_DIM)
                    and f["ids"].shape == (expected_rows,)
                    and f["rows"].shape == (expected_rows,))
    except Exception:
        return False

# ── Main loop ─────────────────────────────────────────────────────────────
out      = Path(OUTPUT_DIR)
manifest = []
total    = 0
t_start  = time.time()

# Resume from existing manifest
manifest_path = out / "manifest.json"
if manifest_path.exists():
    with open(manifest_path) as f:
        old = json.load(f)
        manifest = old.get("shards", [])
        total    = old.get("total_rows", 0)
    print(f"Resuming: {len(manifest)} shards complete, {total:,} rows done")

print(f"\nStreaming {CSV_PATH}")
print(f"BATCH={BATCH_SIZE}  MAX_LEN={MAX_LEN}  DIM={OUT_DIM}  SHARD={SHARD_SIZE:,}")
print(f"Target: {TOTAL_ROWS:,} rows  ({100*total/TOTAL_ROWS:.1f}% complete)\n")

for shard_idx, chunk in enumerate(pd.read_csv(
    CSV_PATH, chunksize=SHARD_SIZE, dtype=CSV_DTYPES, low_memory=False
)):
    n_rows    = len(chunk)
    first_row = total
    last_row  = total + n_rows
    stem      = f"qwen_{first_row:09d}_{last_row:09d}"
    h5_path   = out / f"{stem}.h5"

    # Skip completed shards
    if shard_exists(h5_path, n_rows):
        print(f"[{shard_idx}] SKIP {first_row:,}–{last_row-1:,} (already saved)")
        manifest.append({"shard": shard_idx, "first": first_row,
                         "last": last_row, "rows": n_rows, "file": h5_path.name})
        total = last_row
        continue

    t_shard = time.time()
    print(f"[{shard_idx}] rows {first_row:,}–{last_row-1:,}  ({n_rows:,} movies)")

    # Build texts with itertuples (no dict allocs — the #1 RAM fix)
    texts = [movie_text(r) for r in chunk.itertuples()]
    ids   = chunk["id"].fillna(0).astype("int32").tolist()
    del chunk
    gc.collect()

    arr = embed_shard(texts)
    del texts
    gc.collect()

    # Atomic HDF5 save (write to .tmp then rename — no partial files)
    tmp = h5_path.with_suffix(".tmp.h5")
    with h5py.File(tmp, "w") as hf:
        hf.create_dataset("embeddings", data=arr, dtype="float32",
                          compression="gzip", compression_opts=2)
        hf.create_dataset("ids", data=np.array(ids, dtype="int32"))
        hf.create_dataset("rows", data=np.arange(first_row, last_row, dtype="int32"))
        hf.attrs["model"] = MODEL_ID
        hf.attrs["dim"] = OUT_DIM
        hf.attrs["first_row"] = first_row
        hf.attrs["last_row"] = last_row
        hf.attrs["created"] = datetime.now(timezone.utc).isoformat()
    os.replace(tmp, h5_path)
    del arr, ids
    gc.collect()

    # Per-shard stats
    total   = last_row
    elapsed = time.time() - t_start
    s_time  = time.time() - t_shard
    speed   = n_rows / s_time if s_time > 0 else 0
    if total > 0 and elapsed > 0 and total < TOTAL_ROWS:
        eta_min = (TOTAL_ROWS - total) / (total / elapsed) / 60
    else:
        eta_min = 0
    vram_gb = torch.cuda.memory_allocated() / 1e9
    free_gb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
    pct = 100 * total / TOTAL_ROWS
    print(f"  saved {h5_path.name}")
    print(f"  {speed:.0f} rows/s | {s_time/60:.1f} min/shard | "
          f"done {total:,}/{TOTAL_ROWS:,} ({pct:.1f}%) | "
          f"ETA ~{eta_min:.0f} min | VRAM {vram_gb:.1f}G/{free_gb:.1f}G free\n")

    manifest.append({"shard": shard_idx, "first": first_row,
                     "last": last_row, "rows": n_rows, "file": h5_path.name})
    with open(manifest_path, "w") as f:
        json.dump({"updated_at": datetime.now(timezone.utc).isoformat(),
                   "model": MODEL_ID, "dim": OUT_DIM,
                   "total_rows": total, "target_rows": TOTAL_ROWS,
                   "shards": manifest}, f, indent=2)

elapsed = time.time() - t_start
print(f"\nSession done!  {total:,} movies in {elapsed/3600:.2f} h  ({total/elapsed:.0f} rows/s)")
print(f"Progress: {total:,}/{TOTAL_ROWS:,} ({100*total/TOTAL_ROWS:.1f}%)")

# Size estimate for the zip
h5_size = sum(p.stat().st_size for p in out.glob("*.h5"))
json_size = sum(p.stat().st_size for p in out.glob("*.json"))
print(f"Output size: ~{(h5_size+json_size)/1e6:.0f} MB ({len(manifest)} shards)")
print(f"\n>>> NOW RUN CELL 7 to download your progress!")


## Cell 7 — Download Progress (run before session ends!)

Zips all .h5 shards + manifest.json and triggers browser download. Upload this zip in the next session's Cell 2 to resume.


In [ ]:
import zipfile, os
from pathlib import Path
from google.colab import files

out = Path(OUTPUT_DIR)
zip_path = "/content/embeddings_progress.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(out.iterdir()):
        zf.write(f, arcname=f"embeddings/{f.name}")

size_mb = os.path.getsize(zip_path) / 1e6
n_shards = len(list(out.glob("*.h5")))
print(f"Created embeddings_progress.zip ({size_mb:.0f} MB, {n_shards} shards)")
print("\nDownloading...")
files.download(zip_path)


## Cell 8 — Sanity Check (optional)

Quick validation: embedding norms, NaN check.


In [ ]:
import numpy as np, h5py
from pathlib import Path

out    = Path(OUTPUT_DIR)
shards = sorted(out.glob("qwen_*.h5"))

if shards:
    with h5py.File(shards[0], "r") as hf:
        arr  = hf["embeddings"][:]
        ids  = hf["ids"][:]
        rows = hf["rows"][:]
        print(f"Shards: {len(shards)}  |  First shard: {arr.shape[0]:,} x {arr.shape[1]}")
        print(f"IDs range: {ids.min()} – {ids.max()}  |  Rows: {rows[0]:,} – {rows[-1]:,}")
        print(f"Avg vector norm: {np.linalg.norm(arr[:1000], axis=1).mean():.4f}  (should be ~1.0)")
        print(f"Any NaN/Inf:    {not np.isfinite(arr[:1000]).all()}")
else:
    print("No shards found — run Cell 6 first.")


## Multi-Session Workflow

Colab free gives ~5 hours/day GPU. You'll need multiple sessions:

### Session 1
1. Cells 1 → 2 → 3 → 4 → 5 → 6
2. Let it run until session disconnects or you hit the limit
3. **Cell 7 → download `embeddings_progress.zip` to your computer**

### Sessions 2-N
1. **Upload** `embeddings_progress.zip` to Colab (drag into file panel)
2. Cells 1 → 2 (restores .h5 shards + manifest.json) → 3 (skips if CSV exists) → 4 → 5 → 6
3. Cell 6 auto-skips completed shards and continues where it left off
4. **Cell 7 → download updated zip before session ends**

### Final session
Run Cell 7 one last time. The full output (~1.4M × 768 × float32) is about **4.3 GB** uncompressed, ~2-3 GB zipped.

### Local usage
Drop the `.h5` files into your `embeddings/` folder alongside the TMDB CSV. Then:
```bash
uv run python main.py --similar "mind-bending sci-fi"
```
